# XRF map-averaged spectrum — constrained Voigt fit

Reads a BLISS mesh scan, corrects the MCA spectra for incident flux and detector
dead time, forms the map-averaged spectrum, and fits it with a **constrained**
Voigt model.

## The model

One free amplitude per *line group*, where a group is one (element, shell family)
— e.g. `Nd L`, `Cu K`. That amplitude is the integrated area of the group's most
intense line; every other line of the group is tied to it through its tabulated
relative intensity. Line positions are never free: they come from the tabulated
energies and the fixed calibration. Widths come from a two-parameter detector
resolution law shared by all lines.

So a four-element fit has 4 + 3 + 3 = 10 free parameters covering ~56 lines,
instead of 4 parameters per line.

## Relative intensities

Taken from `xraylib.CS_FluorLine_Kissel`, **not** from `RadRate`. `RadRate` is
normalised within a subshell — the L1, L2 and L3 rates each sum to 1 — so it
carries no information on the relative ionisation of the subshells and cannot be
used across an L series. For Ir it predicts Lb1/La1 = 1.13, against 0.65 from
the Kissel cross-sections and 0.77 measured here.

## Deliberately out of scope for now

* **Energy calibration** is taken as fixed (`E[eV] = 10.032 x channel - 31.263`).
* **No detector efficiency**, so group areas at different energies are not yet
  comparable on an absolute scale.
* **No self-absorption, no secondary fluorescence, no escape or pile-up peaks.**
* **No per-pixel fitting** — that is the next step, and it should be a single
  linear least-squares solve rather than a per-pixel optimiser, because with
  positions and widths fixed the model is linear in the amplitudes.

## 1. Configuration

In [ ]:
# %matplotlib widget      # interactive pan/zoom; needs ipympl
%matplotlib inline

import json
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from bliss_hdf5_reader import BlissMeshScan
from xrf_beamline import BM32_WHITE_BEAM
from xrf_fit import (Calibration, Detector, element_line_groups, normalise_spectra, fit_map,
                     plot_pixel_fits, select_pixels,
                     average_spectrum, counting_error, fit_average_spectrum,
                     plot_fit, identify_peaks, snip_background)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ---------------------------------------------------------------- input data
# At the beamline, paths look like
#   /data/visitor/<proposal>/bm32/<date>/RAW_DATA/<sample>/<sample>_<scan>/<...>.h5
SAMPLES = {
    "Nd2Ir2O7_bulk": dict(
        h5=Path("NIO295_laue_map_precise.h5"), scan=1,
        elements=["Nd", "Ir", "Pt", "Pb"],     # Pb is beamline contamination
        roi_ev=(4300.0, 15800.0), bkg_order=2,
        matrix="Nd2Ir2O7", thickness_nm=None,
        note="bulk single crystal — self-absorption is significant",
    ),
    "InGaAs_film": dict(
        h5=Path("InPoSi_v2_center.h5"), scan=1,
        elements=["In", "Ga", "As", "Cu", "Ni"],   # Cu, Ni are contamination
        roi_ev=(2900.0, 12600.0), bkg_order=3,
        matrix="In0.5Ga0.5As", thickness_nm=200.0,
        note="thin film on Si — self-absorption is negligible (~1 % for 200 nm)",
    ),
}
SAMPLE = "InGaAs_film"          # <- pick the sample here

_s = SAMPLES[SAMPLE]
H5_FILE     = _s["h5"]
SCAN_NUMBER = _s["scan"]
MCA         = "xia_det0"
MONITOR     = "mon"
LIVE_TIME   = f"{MCA}_live_time"
print(f"sample: {SAMPLE} — {_s['note']}")

OUT_DIR = Path("processed")                        # at the beamline: .../PROCESSED_DATA
OUT_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------ physics setup
CALIB = Calibration(gain=10.032, offset=-31.263)   # E[eV] = gain*channel + offset

# Quoted BM32 XIA resolution is 125-150 eV FWHM; 135 eV at Mn Ka (5.9 keV) sets
# the electronic-noise term for a Si detector at the stated Fano factor.
DETECTOR = Detector.from_fwhm(fwhm_ev=135.0, at_energy_ev=5900.0,
                              material="Si", fano=0.12, gamma_ev=5.0)

ELEMENTS   = _s["elements"]
FAMILIES   = ("K", "L")
# Excitation weighting. USE_WHITE_BEAM=True is the physically correct treatment
# for a polychromatic beam, but see section 6: on its own it makes the fit worse,
# because it is only one of three energy-dependent factors and the other two
# (self-absorption, detector efficiency) are not implemented yet and push the
# opposite way. Left off by default, deliberately.
USE_WHITE_BEAM = False
E0_EFF_KEV = 20.0        # effective excitation energy used when USE_WHITE_BEAM is False;
                         # must exceed the highest edge of interest (Ir L1 = 13.42 keV)
E_GRID_KEV = np.linspace(3.0, 40.0, 500)   # energy grid for the beamline model

# Incident air path. This is the dominant low-energy filter on the line: 3 m of
# ambient air transmits ~1e-7 at 5 keV and 12 % at 10 keV, so it, not the Be
# windows, sets the lower edge of the usable band. NOT the same thing as the
# sample-to-detector air path, which attenuates the emitted lines and belongs
# to the (not yet implemented) detector response.
AIR_PATH_M   = 3.0
AIR_TEMP_C   = 20.0
AIR_PRESS_HPA = 1013.25    # local pressure; ~990 hPa at ESRF altitude
MIN_REL    = 0.005       # drop lines below this fraction of their group reference
ROI_EV     = _s["roi_ev"]
BKG_ORDER  = _s["bkg_order"]   # polynomial background order over the ROI

print(f"detector: noise = {DETECTOR.noise_ev:.1f} eV, Fano = {DETECTOR.fano}, "
      f"gamma = {DETECTOR.gamma_ev} eV")
for e in (5900, 9170, 12500):
    print(f"  FWHM({e/1000:5.2f} keV) = {DETECTOR.fwhm_ev(e):6.1f} eV")

## 2. Load the scan

In [ ]:
scan = BlissMeshScan(h5_filepath=H5_FILE, scan_number=SCAN_NUMBER, verbose=False)
scan.read_data(extra_counters=[MCA, MONITOR, LIVE_TIME, f"{MCA}_fractional_dead_time"])

print(scan.get_summary())

spectra = scan.counters[MCA].astype(float)        # (n_pixels, n_channels)
n_pixels, n_channels = spectra.shape
print(f"\nMCA: {n_pixels} pixels x {n_channels} channels, {spectra.sum():.3e} counts total")

# Two warnings above are expected and harmless on a fluorescence-only analysis:
#   - the eiger image folder is absent (no Laue images needed here);
#   - the 2D MCA array is skipped by the 1D grid reshaper.

## 3. Normalisation: incident flux and detector dead time

Both corrections are per pixel and multiplicative:
`y_corrected = y_raw / (monitor_rel * live_time)`.

Dividing by the XIA live time converts recorded counts into a true incoming
rate. On white-beam maps this is the larger of the two corrections by far, and
it is spatially structured — it tracks total scattering power, so it does **not**
cancel in an elemental ratio.

In [ ]:
monitor   = scan.counters[MONITOR]
live_time = scan.counters[LIVE_TIME]
fdt       = scan.counters[f"{MCA}_fractional_dead_time"]

corrected, scale = normalise_spectra(spectra, monitor=monitor, live_time=live_time)

print(f"monitor        : {monitor.min():.4g} - {monitor.max():.4g} "
      f"(spread {100*np.ptp(monitor)/monitor.mean():.1f} %)")
print(f"live time (s)  : {live_time.min():.4g} - {live_time.max():.4g}")
print(f"dead-time frac.: {fdt.min():.3f} - {fdt.max():.3f}, median {np.median(fdt):.3f}")
print(f"-> correction factor spans {1/(1-fdt.max()):.2f}x across the map")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (data, title) in zip(axes, [(monitor, "monitor"),
                                    (live_time, "XIA live time (s)"),
                                    (fdt, "fractional dead time")]):
    im = ax.imshow(data.reshape(scan.scan_shape), cmap="viridis", origin="lower")
    ax.set_title(title)
    ax.set_xlabel(f"{scan.fast_motor_name} (pixel)")
    ax.set_ylabel(f"{scan.slow_motor_name} (pixel)")
    fig.colorbar(im, ax=ax)
fig.suptitle("Per-pixel normalisation terms", y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{H5_FILE.stem}_normalisation.png", dpi=150, bbox_inches="tight")

## 4. Map-averaged spectrum

Two uncertainty estimates are computed, and they answer different questions:

* **counting error** — the Poisson error of the average, propagated through the
  normalisation. This is the right weight for *fitting* the average spectrum.
* **standard error of the mean** — the pixel-to-pixel spread divided by sqrt(N).
  Much larger than the counting error wherever the map is genuinely
  heterogeneous, so it is a map-structure diagnostic, not a fit weight.

In [ ]:
avg, sem, n_used = average_spectrum(corrected)
cnt_err = counting_error(spectra, scale)

lo, hi = CALIB.channel_range(*ROI_EV, n_channels)
ROI = (lo, hi)
print(f"pixels used: {n_used}/{n_pixels}")
print(f"ROI: channels {lo}-{hi} = {CALIB.energy(lo)/1000:.2f}-{CALIB.energy(hi-1)/1000:.2f} keV")
print(f"median counting error / peak amplitude = {np.median(cnt_err[lo:hi])/avg[lo:hi].max():.2e}")
print(f"median SEM            / peak amplitude = {np.median(sem[lo:hi])/avg[lo:hi].max():.2e}")

ch = np.arange(n_channels)
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.fill_between(CALIB.energy(ch)/1000, avg - sem, avg + sem, color="C0", alpha=0.3,
                lw=0, label=r"pixel-to-pixel spread ($\pm$ SEM)")
ax.plot(CALIB.energy(ch)/1000, avg, "k-", lw=0.9, label="map-averaged spectrum")
ax.axvspan(CALIB.energy(lo)/1000, CALIB.energy(hi-1)/1000, color="0.9", zorder=0, label="fit ROI")
ax.set_yscale("log")
ax.set_xlabel("Energy (keV)")
ax.set_ylabel("Counts per live second (monitor-normalised)")
ax.set_xlim(0, CALIB.energy(n_channels-1)/1000)
ax.set_ylim(max(avg[avg > 0].min(), 1e-3), avg.max()*3)
ax.legend()
ax.set_title(f"{H5_FILE.stem} — scan {SCAN_NUMBER}")
fig.tight_layout()
fig.savefig(OUT_DIR / f"{H5_FILE.stem}_average_spectrum.png", dpi=150, bbox_inches="tight")

## 5. Which elements? — peak scouting

`identify_peaks` lists tabulated lines whose energy is close to each detected
peak, ranked by radiative rate. It is a *scouting* tool: a close energy match is
suggestive, never proof. Confirmation comes from the fit — a genuine element
accounts for its whole line series at the tabulated ratios, a coincidence does
not.

In [ ]:
for p in identify_peaks(avg, CALIB, ROI, prominence_frac=0.01):
    print(f"ch {p['channel']:4d}  {p['energy_ev']/1000:6.3f} keV  h={p['height']:8.1f}")
    for c in p["candidates"]:
        print(f"        {c}")

## 6. White-beam excitation

The beamline is modelled as a bending-magnet source propagated through its
optics (`xrf_beamline.py`), giving the incident spectrum Phi(E) at the sample.
The line strengths then become flux-weighted integrals,
`r_l = int sigma_l(E) Phi(E) dE`, instead of cross-sections at one energy.

This is not cosmetic. Each L subshell is excited only by the part of the
spectrum above *its own* edge, and for the heavy elements here those edges sit
inside the band (Ir L3/L2/L1 at 11.22/12.82/13.42 keV; Pb at 13.04/15.20/15.86).
Note also that the mirrors are **Ir coated**, so the beam carries the Ir L edges
of the optics at exactly the energies where the sample's Ir L shells are excited
- visible as steps in Phi(E).

In [ ]:
beamline = BM32_WHITE_BEAM(air_path_m=AIR_PATH_M, temperature_c=AIR_TEMP_C,
                           pressure_hpa=AIR_PRESS_HPA)
raw_flux, phi, transfers = beamline.spectrum(E_GRID_KEV)
e_lo, e_hi = beamline.band_limits(E_GRID_KEV, frac=0.10)

print(f"critical energy      : {beamline.source.critical_energy_kev:.2f} keV")
print(f"band at 10 % of peak : {e_lo:.2f} - {e_hi:.2f} keV")
print(f"band at  1 % of peak : {beamline.band_limits(E_GRID_KEV, 0.01)[0]:.2f} - "
      f"{beamline.band_limits(E_GRID_KEV, 0.01)[1]:.2f} keV")
print(f"flux peaks at        : {E_GRID_KEV[np.argmax(phi)]:.2f} keV")

# The low-energy edge is a sensitive measure of the air path, so a quoted band
# edge can be inverted for the path length. Useful as a consistency check.
from scipy.optimize import brentq
for target in (5.0, 6.0, 7.0):
    try:
        L = brentq(lambda x: BM32_WHITE_BEAM(air_path_m=x, temperature_c=AIR_TEMP_C,
                                             pressure_hpa=AIR_PRESS_HPA
                                             ).band_limits(E_GRID_KEV, 0.01)[0] - target,
                   0.02, 8.0, xtol=1e-3)
        print(f"  a 1 % band edge at {target:.1f} keV would imply {L:.2f} m of air")
    except ValueError:
        pass

from scipy.integrate import simpson
total = simpson(phi, x=E_GRID_KEV)
print("\nfraction of incident flux above each L edge:")
for el in ELEMENTS:
    z = __import__("xraylib").SymbolToAtomicNumber(el)
    xl = __import__("xraylib")
    parts = []
    for shell, nm in [(xl.L3_SHELL, "L3"), (xl.L2_SHELL, "L2"), (xl.L1_SHELL, "L1")]:
        edge = xl.EdgeEnergy(z, shell)
        m = E_GRID_KEV >= edge
        parts.append(f"{nm} ({edge:5.2f} keV) {100*simpson(phi[m], x=E_GRID_KEV[m])/total:5.1f} %")
    print(f"  {el}: " + "   ".join(parts))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1], "hspace": 0.1})
ax1.plot(E_GRID_KEV, raw_flux/raw_flux.max(), "k--", lw=1.3, label="bending-magnet source")
ax1.plot(E_GRID_KEV, phi/raw_flux.max(), "r-", lw=1.8, label="at sample, after optics")
ax1.axvspan(e_lo, e_hi, color="C1", alpha=0.12, zorder=0, label=f"10 % band: {e_lo:.1f}-{e_hi:.1f} keV")
ax1.axvline(beamline.source.critical_energy_kev, color="C0", ls=":", lw=1.2,
            label=f"$E_c$ = {beamline.source.critical_energy_kev:.1f} keV")
ax1.set_ylabel("Flux (normalised to source peak)")
ax1.set_title("Incident white-beam spectrum")
ax1.legend(fontsize=9)
for lab, t in transfers:
    ax2.plot(E_GRID_KEV, t, lw=1.1, label=lab)
ax2.set_xlabel("Energy (keV)"); ax2.set_ylabel("Reflectivity /\ntransmission")
ax2.set_yscale("log"); ax2.set_ylim(1e-8, 2.0)   # the air path spans ~7 decades
ax2.legend(fontsize=7, ncol=2)
fig.savefig(OUT_DIR / "BM32_white_beam_spectrum.png", dpi=150, bbox_inches="tight")

In [ ]:
# How much does flux weighting change the tabulated ratios?
g_scalar = element_line_groups(ELEMENTS, *ROI_EV, e0_eff_kev=E0_EFF_KEV,
                               families=FAMILIES, min_rel=MIN_REL)
g_white  = element_line_groups(ELEMENTS, *ROI_EV, families=FAMILIES, min_rel=MIN_REL,
                               incident_spectrum=(E_GRID_KEV, phi))

print(f"{'group':<7}{'line':<7}{'E (eV)':>9}{'scalar':>9}{'white beam':>12}{'change':>9}")
for gs, gw in zip(g_scalar, g_white):
    ref = {l.name: l.rel for l in gs.lines}
    for l in gw.lines:
        if l.name in ref and max(l.rel, ref[l.name]) > 0.05:
            print(f"{gw.label:<7}{l.name:<7}{l.energy_ev:>9.0f}{ref[l.name]:>9.3f}"
                  f"{l.rel:>12.3f}{100*(l.rel/ref[l.name] - 1):>8.1f} %")

print("\nL3 ratios are unchanged by construction (the reference line is an L3 line);")
print("L2 and L1 lines are the ones that move.")

In [ ]:
# ...and what does that do to the fit?
for tag, grp in [("scalar E0", g_scalar), ("white beam", g_white)]:
    r = fit_average_spectrum(avg, grp, CALIB, DETECTOR, ROI, yerr=cnt_err,
                             background_order=BKG_ORDER)
    rr = np.abs(r.residual) / r.y.max()
    print(f"{tag:<12} chi2_red={r.chi2_red:8.1f}  resid rms={100*np.sqrt((rr**2).mean()):5.2f} %  "
          + "  ".join(f"{k}={v:.4g}" for k, v in r.areas.items()))

# Flux weighting can only change ratios WITHIN a multi-subshell series. A K series
# has a single initial vacancy, so its line ratios are independent of the incident
# spectrum -- for a K-only element list this section changes nothing at all.
import xraylib
moved = [(gs.label, l.name, d, l.rel) for gs, gw in zip(g_scalar, g_white)
         for l in gw.lines for d in [dict((x.name, x.rel) for x in gs.lines).get(l.name)]
         if d and abs(l.rel/d - 1) > 0.02]
if not moved:
    print("\nNo line ratio moved by more than 2 % -> every group here is a K series,")
    print("so Phi(E) cannot affect this fit. Only multi-subshell (L, M) series respond.")
else:
    print(f"\n{len(moved)} line ratios moved by >2 %; the largest:")
    for lab, nm, d, w in sorted(moved, key=lambda t: -abs(t[3]/t[2]-1))[:5]:
        print(f"   {lab:<7}{nm:<7}{d:.4f} -> {w:.4f}  ({100*(w/d-1):+.1f} %)")

# Whether the excitation side can be improved in isolation depends on sample
# thickness, because the depth integral does not factorise for a thick sample:
#     I_l ~ int dE Phi(E) sigma_l(E) / ( mu(E)/sin(psi_in) + mu(E_l)/sin(psi_out) )
# mu(E) sits INSIDE the energy integral, so harder photons -- the ones that
# preferentially excite the higher subshells -- are up-weighted by penetrating
# deeper. For a thin film the denominator becomes constant and the simple
# flux-weighted product is recovered.
MATRIX = _s["matrix"]
t_nm = _s["thickness_nm"]
pairs = [(g.lines[0].energy_ev, g.lines[1].energy_ev, g.label)
         for g in g_scalar if len(g.lines) > 1][:3]
print(f"\nmatrix {MATRIX}: attenuation ratio for the two strongest lines of each group")
print("(this brackets how much self-absorption can enhance an observed ratio)")
for e_lo, e_hi, lab in pairs:
    lo, hi = min(e_lo, e_hi)/1000, max(e_lo, e_hi)/1000
    m_lo, m_hi = xraylib.CS_Total_CP(MATRIX, lo), xraylib.CS_Total_CP(MATRIX, hi)
    print(f"  {lab:<7}{lo:6.3f}/{hi:6.3f} keV  mu = {m_lo:6.1f}/{m_hi:6.1f} cm2/g"
          f"  -> max enhancement {m_lo/m_hi:.3f}")
if t_nm is not None:
    rho = 5.5   # g/cm3, approximate for the film material
    print(f"\n  optical depth of a {t_nm:.0f} nm film (rho = {rho} g/cm3):")
    for e_lo, e_hi, lab in pairs:
        mu0 = xraylib.CS_Total_CP(MATRIX, min(e_lo, e_hi)/1000)
        print(f"    {lab:<7}mu*rho*t = {mu0*rho*t_nm*1e-7:.4f}"
              f"  -> self-absorption at the {100*mu0*rho*t_nm*1e-7:4.1f} % level")
    print("  The K lines are effectively free of it; the soft L lines are not.")
    print("  With the denominator near-constant the factorised flux weighting is valid here.")
else:
    print("  bulk sample: the coupled depth integral above is required; the factorised")
    print("  flux weighting is NOT valid here (see the session notes).")

## 7. Build the line groups

Each group's lines are listed relative to its most intense line (`rel = 1`),
which is the one carrying the free amplitude.

In [ ]:
groups = g_white if USE_WHITE_BEAM else g_scalar
print("excitation weighting:", "flux-weighted over Phi(E)" if USE_WHITE_BEAM
      else f"single effective energy E0 = {E0_EFF_KEV} keV")

for g in groups:
    print(f"\n{g.label}  ({len(g.lines)} lines, sum of relative intensities = {g.total_rel:.3f})")
    print(f"  {'line':<8}{'E (eV)':>10}{'rel':>9}")
    for line in g.lines:
        print(f"  {line.name:<8}{line.energy_ev:>10.0f}{line.rel:>9.3f}")

n_free = len(groups) + (BKG_ORDER + 1) + 3
print(f"\n{sum(len(g.lines) for g in groups)} lines modelled with {n_free} free parameters")

## 8. Fit

In [ ]:
result = fit_average_spectrum(avg, groups, CALIB, DETECTOR, ROI,
                              yerr=cnt_err, background_order=BKG_ORDER,
                              fit_resolution=True, fit_gamma=True)
print(result.summary())

resid_rel = np.abs(result.residual) / result.y.max()
resid_loc = np.abs(result.residual) / np.maximum(result.y, 1e-12)
print(f"\nresidual vs PEAK amplitude : rms {100*np.sqrt((resid_rel**2).mean()):.2f} %, "
      f"max {100*resid_rel.max():.2f} %")
print(f"residual vs LOCAL data     : median {100*np.median(resid_loc):.1f} %")
# The second number is the honest one for weak lines. A spectrum with one
# dominant peak can show a tiny rms/peak while the model is badly wrong in the
# low-signal regions where a trace element sits.

A note on reduced chi-squared. Averaging thousands of pixels makes the counting
error on the mean spectrum ~0.01 % of the peak amplitude, so any model
imperfection at the per-cent level gives chi2_red in the thousands. Here it
measures **model inadequacy, not noise**, and the parameter errors it reports are
correspondingly optimistic. Judge the fit on the residual as a fraction of peak
amplitude, and on whether the residual is structured.

In [ ]:
fig, (ax, axr) = plot_fit(result, energy_axis=True, logy=True)
fig.savefig(OUT_DIR / f"{H5_FILE.stem}_voigt_fit.png", dpi=150, bbox_inches="tight")

## 9. How much does the element list matter?

Overlapping series compete for the same counts, so dropping one element biases
the others. This leave-one-out test quantifies that — it is the honest way to
report how well-determined each area is, and it matters more than the formal fit
errors.

In [ ]:
ref = result.areas
rows = []
for drop in ELEMENTS:
    kept = [e for e in ELEMENTS if e != drop]
    g2 = element_line_groups(kept, *ROI_EV, e0_eff_kev=E0_EFF_KEV,
                             families=FAMILIES, min_rel=MIN_REL)
    r2 = fit_average_spectrum(avg, g2, CALIB, DETECTOR, ROI, yerr=cnt_err,
                              background_order=BKG_ORDER)
    rr = np.abs(r2.residual) / r2.y.max()
    shifts = ", ".join(f"{k} {100*(r2.areas[k]-ref[k])/ref[k]:+.1f} %" for k in r2.areas)
    rows.append((drop, r2.chi2_red, 100*np.sqrt((rr**2).mean()), shifts))

print(f"{'dropped':<10}{'chi2_red':>12}{'resid rms %':>14}   shift in remaining areas")
print(f"{'(none)':<10}{result.chi2_red:>12.1f}"
      f"{100*np.sqrt((resid_rel**2).mean()):>14.2f}   -")
for drop, c2, rms, shifts in rows:
    print(f"{drop:<10}{c2:>12.1f}{rms:>14.2f}   {shifts}")

## 10. Save

In [ ]:
stem = H5_FILE.stem
np.savez_compressed(
    OUT_DIR / f"{stem}_average_spectrum.npz",
    channel=np.arange(n_channels), energy_ev=CALIB.energy(np.arange(n_channels)),
    average=avg, counting_error=cnt_err, sem=sem,
    roi=np.array(ROI), fit_channels=result.channels, fit_model=result.yfit,
    background=result.model.background(result.popt),
    **{f"component_{g.label.replace(' ', '')}": result.model.group_profile(i, result.popt)
       for i, g in enumerate(result.model.groups)},
)

record = {
    "file": str(H5_FILE), "scan": SCAN_NUMBER, "pixels_used": n_used,
    "calibration": {"gain_ev_per_channel": CALIB.gain, "offset_ev": CALIB.offset},
    "detector": {"material": DETECTOR.material, "noise_ev": DETECTOR.noise_ev,
                 "fano": DETECTOR.fano, "gamma_ev": DETECTOR.gamma_ev},
    "elements": ELEMENTS, "families": list(FAMILIES), "e0_eff_kev": E0_EFF_KEV,
    "min_rel": MIN_REL, "roi_channels": list(ROI), "background_order": BKG_ORDER,
    "chi2_red": result.chi2_red,
    "residual_rms_percent_of_peak": float(100*np.sqrt((resid_rel**2).mean())),
    "parameters": {k: {"value": v, "stderr": e} for k, (v, e) in result.params().items()},
    "group_areas": {g.label: {"reference_line": g.reference.name,
                              "reference_area": result.areas[g.label],
                              "reference_area_stderr": result.area_errors[g.label],
                              "total_area": result.areas[g.label]*g.total_rel,
                              "n_lines": len(g.lines)}
                    for g in result.model.groups},
}
(OUT_DIR / f"{stem}_fit.json").write_text(json.dumps(record, indent=2))
print("written:", *(p.name for p in sorted(OUT_DIR.glob(f"{stem}*"))), sep="\n  ")

## 11. Per-pixel maps

With the peak positions and widths fixed by the average-spectrum fit, the model
is **linear** in the group amplitudes and the background coefficients. The whole
map is therefore one stacked linear solve: assemble `A^T W A` (only
`n_components x n_components` per pixel) and solve all pixels at once. The
per-pixel covariance comes out of the same factorisation, so every area map
arrives with an error map.

Weights are Poisson on the raw counts, `Var(y) = raw / scale^2`. Without them
the strongest channels dominate each pixel and the reported errors are
meaningless.

In [ ]:
import time
t0 = time.time()
maps = fit_map(spectra, scale, result.model, result.popt, scan.scan_shape)
print(f"solved {spectra.shape[0]} pixels x {ROI[1]-ROI[0]} channels x "
      f"{len(maps.group_labels)+result.model.n_bkg} components in {time.time()-t0:.2f} s")
print(f"\nper-pixel reduced chi2: median {np.nanmedian(maps.chi2_red):.2f}  "
      f"(5-95 %: {np.nanpercentile(maps.chi2_red,5):.2f} - {np.nanpercentile(maps.chi2_red,95):.2f})")
print("A median near 1 is the real validation of the constrained model: it says the")
print("fixed line ratios and widths describe an INDIVIDUAL spectrum at the noise level.\n")
for lab in maps.group_labels:
    a, e = maps.area_map(lab, total=True), maps.error_map(lab, total=True)
    print(f"  {lab:<6} median {np.nanmedian(a):11.4g}   median S/N {np.nanmedian(a/e):7.1f}"
          f"   negative pixels {100*np.nanmean(a < 0):5.2f} %")

### Fit quality, group by group

A median per-pixel chi2 near 1 says the model is right *where the counts are*.
It says nothing about a group contributing a fraction of a per-cent of the
spectrum. `quality_report` separates the two: a group whose per-pixel S/N is of
order a few is detected but not quantifiable, and its map is noise.

In [ ]:
print(maps.quality_report())

### Individual pixel fits

Averaged residuals and a median chi2 both hide what a single spectrum looks
like. These panels are the direct check: whether the fixed line ratios hold
pixel by pixel, whether the background is plausible at the local count level,
and where the weak groups vanish into the noise.

Use `mode="worst"` when something looks wrong on a map — it shows the pixels
with the largest reduced chi2, which is where a missing element, a bad
background order or a pile-up artifact shows up first. `mode="spread"` walks the
chi2 distribution in equal quantiles for a representative rather than typical
sample.

In [ ]:
fig, axes, pixels = plot_pixel_fits(spectra, scale, result.model, maps, result.popt,
                                    n=10, mode="random", seed=0, ncols=5)
fig.savefig(OUT_DIR / f"{H5_FILE.stem}_pixel_fits_random.png", dpi=140, bbox_inches="tight")

# the diagnostic view: the ten worst-fitting pixels on the map
fig_w, _, pixels_w = plot_pixel_fits(spectra, scale, result.model, maps, result.popt,
                                     n=10, mode="worst", ncols=5)
fig_w.savefig(OUT_DIR / f"{H5_FILE.stem}_pixel_fits_worst.png", dpi=140, bbox_inches="tight")
print("random pixels (slow, fast):", [(int(i)//maps.shape[1], int(i)%maps.shape[1]) for i in pixels])
print("worst  pixels (slow, fast):", [(int(i)//maps.shape[1], int(i)%maps.shape[1]) for i in pixels_w])

### Quantitative ratio

For two elements A and B measured through the same spectrum,

    n_A / n_B  =  (I_A / I_B) * (S_B * W_B * eps_B) / (S_A * W_A * eps_A)

with `S` the flux-weighted cross-section sum `int sigma(E) Phi(E) dE` over the
group's lines, `W` the atomic weight (converting mass to molar cross-sections)
and `eps` the detection efficiency at the emission energies.

Self-absorption is omitted, which is legitimate here only because both lines are
hard: Ga Ka and As Ka have 1/e escape lengths of 21.5 and 18.9 um in this
matrix, so over any plausible layer thickness their *ratio* barely moves. It is
**not** legitimate for a soft line — see the closure test below.

`eps` needs the detector, which is the remaining unknown; the loop below reports
the systematic spread over plausible configurations.

In [ ]:
from scipy.integrate import simpson
import xraylib
from xrf_beamline import AIR_FORMULA, air_density

SHELL_OF = {"K": xraylib.K_SHELL, "L1": xraylib.L1_SHELL,
            "L2": xraylib.L2_SHELL, "L3": xraylib.L3_SHELL}

def flux_weighted_strength(element, lines, energy_kev, flux):
    """Sum of int sigma_l(E) Phi(E) dE over a group's lines, plus the per-line terms."""
    z = xraylib.SymbolToAtomicNumber(element)
    total, per_line = 0.0, []
    for l in lines:
        macro = getattr(xraylib, l.name + "_LINE")
        key = "K" if l.name.startswith("K") else l.name[:2]
        edge = xraylib.EdgeEnergy(z, SHELL_OF[key])
        cs = np.array([xraylib.CS_FluorLine_Kissel(z, macro, float(e)) if e > edge else 0.0
                       for e in energy_kev])
        s = simpson(cs * flux, x=energy_kev)
        total += s
        per_line.append((l.energy_ev, s))
    return total, per_line

def detector_efficiency(E_kev, window_um=25.0, window="Be", window_rho=1.85,
                        air_cm=3.0, active_um=450.0, active="Si", active_rho=2.33):
    """Transmission of the detector window and detection air path, times the
    absorption probability in the active volume."""
    t_win = np.exp(-xraylib.CS_Total_CP(window, E_kev) * window_rho * window_um * 1e-4)
    t_air = np.exp(-xraylib.CS_Total_CP(AIR_FORMULA, E_kev) * air_density() * air_cm)
    absorbed = 1.0 - np.exp(-xraylib.CS_Total_CP(active, E_kev) * active_rho * active_um * 1e-4)
    return t_win * t_air * absorbed

def group_efficiency(per_line, **kw):
    """Strength-weighted mean efficiency across a group's lines."""
    return (sum(s * detector_efficiency(E / 1000.0, **kw) for E, s in per_line)
            / sum(s for _, s in per_line))

def number_ratio(label_a, label_b, el_a, el_b, maps, flux, **det_kw):
    ga = next(g for g in result.model.groups if g.label == label_a)
    gb = next(g for g in result.model.groups if g.label == label_b)
    Sa, pla = flux_weighted_strength(el_a, ga.lines, E_GRID_KEV, flux)
    Sb, plb = flux_weighted_strength(el_b, gb.lines, E_GRID_KEV, flux)
    k = ((Sb * xraylib.AtomicWeight(xraylib.SymbolToAtomicNumber(el_b))
          * group_efficiency(plb, **det_kw))
         / (Sa * xraylib.AtomicWeight(xraylib.SymbolToAtomicNumber(el_a))
            * group_efficiency(pla, **det_kw)))
    r, dr = maps.ratio_map(label_a, label_b)
    return r * k, dr * k, k

DETECTORS = {
    "Si 300 um":  dict(active_um=300),
    "Si 450 um":  dict(active_um=450),
    "Si 1000 um": dict(active_um=1000),
    "Ge 5 mm":    dict(active="Ge", active_rho=5.323, active_um=5000),
}
print(f"{'detector':<12}{'k factor':>11}{'median n_Ga/n_As':>19}")
for lab, kw in DETECTORS.items():
    xm, dxm, k = number_ratio("Ga K", "As K", "Ga", "As", maps, phi, **kw)
    print(f"{lab:<12}{k:>11.4f}{np.nanmedian(xm):>19.4f}")

x_map, dx_map, K_GA_AS = number_ratio("Ga K", "As K", "Ga", "As", maps, phi,
                                      **DETECTORS["Si 450 um"])
v = x_map[np.isfinite(x_map)]
sig_stat = np.nanmedian(dx_map) * np.sqrt(np.nanmedian(maps.chi2_red))
print(f"\nnominal (Si 450 um): n_Ga/n_As = {np.median(v):.4f}")
print(f"  statistical error per pixel {sig_stat:.4f}; map spread {v.std():.4f}"
      f" -> excess (real) variation {np.sqrt(max(v.std()**2 - sig_stat**2, 0)):.4f}")

### Closure test — and why the In content is NOT quantitative here

Two independent reasons, both fatal for an absolute In number:

1. **The signal is far too weak.** In L carries a per-pixel S/N of order 5-10
   against ~150-200 for Ga K and As K, on a local background of a few counts per
   live second. The per-pixel In map is noise; even the map-averaged area is only
   a detection.
2. **It samples a different depth.** In La at 3.4 keV has a 1/e escape length of
   ~3.0 um in this matrix against ~21 um for Ga Ka. The lines do not probe the
   same volume, so the thin-film approximation that makes Ga/As reliable fails
   for In L specifically.

On top of that the low-energy end sits on the worst-modelled part of the
background, and the detector-window and air-path attenuation that suppress a
3.4 keV line are large and currently uncalibrated.

Treat `n_In/n_As` below as a **lower bound and an order of magnitude at best**.
The Ga/As ratio is unaffected by all of this: both are hard K lines of similar
energy, and the quantities that matter cancel.

In [ ]:
if {"In L", "Ga K", "As K"} <= set(maps.group_labels):
    i_map, _, _ = number_ratio("In L", "As K", "In", "As", maps, phi, **DETECTORS["Si 450 um"])
    n_ga, n_in = np.nanmedian(x_map), np.nanmedian(i_map)
    print(f"n_Ga/n_As = {n_ga:.4f}   n_In/n_As = {n_in:.4f}   sum = {n_ga+n_in:.4f}"
          f"   (should be 1.000)")
    MATF, RHOF = _s["matrix"], 5.5
    print(f"\n1/e escape length in {MATF} (rho {RHOF} g/cm3):")
    for lab, E in [("In La", 3.410), ("Ga Ka", 9.368), ("As Ka", 10.687)]:
        mu = xraylib.CS_Total_CP(MATF, E)
        print(f"  {lab:<7}{E:6.3f} keV   mu = {mu:6.1f} cm2/g   {1e4/(mu*RHOF):6.2f} um")
    deficit = (1.0 - n_ga) / n_in
    mu_in = xraylib.CS_Total_CP(MATF, 3.410)
    t_need = np.log(deficit) / (mu_in * RHOF) * 1e4
    mu_ga, mu_as = xraylib.CS_Total_CP(MATF, 9.368), xraylib.CS_Total_CP(MATF, 10.687)
    print(f"\nIn is low by x{deficit:.1f}; pure absorption would need {t_need:.1f} um of overlayer.")
    print(f"Across that thickness the Ga/As RATIO changes by only "
          f"{100*(np.exp(-(mu_ga-mu_as)*RHOF*t_need*1e-4)-1):+.1f} % "
          f"-- which is why Ga/As survives and In/As does not.")

In [ ]:
ga_map, as_map = maps.area_map("Ga K", total=True), maps.area_map("As K", total=True)
ratio, dratio = maps.ratio_map("Ga K", "As K")
fast, slow = scan.fast_motor_name, scan.slow_motor_name
XX = (scan.counters[fast] - scan.counters[fast].min()).reshape(scan.scan_shape) * 1000.0
YY = (scan.counters[slow] - scan.counters[slow].min()).reshape(scan.scan_shape) * 1000.0

fig, axes = plt.subplots(2, 3, figsize=(15.5, 8.6))
def panel(ax, Z, title, cmap="viridis", cbl="", **kw):
    im = ax.pcolormesh(XX, YY, Z, cmap=cmap, shading="nearest", **kw)
    ax.set_aspect("equal"); ax.set_title(title, fontsize=11)
    ax.set_xlabel(f"{fast} (µm)"); ax.set_ylabel(f"{slow} (µm)")
    fig.colorbar(im, ax=ax, label=cbl, fraction=0.046)

panel(axes[0,0], ga_map, "Ga K total area", cbl="counts / live s (norm.)")
panel(axes[0,1], as_map, "As K total area", cbl="counts / live s (norm.)")
panel(axes[0,2], ratio, "Ga K / As K intensity ratio", cmap="inferno", cbl="ratio")
lo, hi = np.nanpercentile(x_map, [1, 99])
panel(axes[1,0], x_map, r"$n_{\rm Ga}/n_{\rm As}$", cmap="cividis", cbl=r"$n_{\rm Ga}/n_{\rm As}$",
      vmin=lo, vmax=hi)
ax = axes[1,1]
ax.hist(v, bins=60, color="steelblue", edgecolor="black", lw=0.4, alpha=0.85)
ax.axvline(np.median(v), color="red", ls="--", lw=1.5, label=f"median {np.median(v):.4f}")
ax.set_xlabel(r"$n_{\rm Ga}/n_{\rm As}$"); ax.set_ylabel("pixels"); ax.legend(fontsize=9)
ax.set_title(f"spread {v.std():.4f}, statistical {sig_stat:.4f}", fontsize=10)
panel(axes[1,2], maps.chi2_red.reshape(scan.scan_shape), r"per-pixel reduced $\chi^2$",
      cmap="magma", cbl=r"$\chi^2_\nu$", vmin=0.9, vmax=1.4)
fig.suptitle(f"{SAMPLE} — per-pixel constrained Voigt fit, "
             f"{scan.scan_shape[0]}x{scan.scan_shape[1]} pixels", y=0.98)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{H5_FILE.stem}_elemental_maps.png", dpi=150, bbox_inches="tight")

np.savez_compressed(OUT_DIR / f"{H5_FILE.stem}_maps.npz",
                    **{f"area_{l.replace(' ','')}": maps.area_map(l, total=True) for l in maps.group_labels},
                    **{f"err_{l.replace(' ','')}": maps.error_map(l, total=True) for l in maps.group_labels},
                    ratio_Ga_As=ratio, ratio_err=dratio, n_Ga_over_n_As=x_map,
                    chi2_red=maps.chi2_red.reshape(scan.scan_shape),
                    k_factor=np.array([K_GA_AS]), x_um=XX, y_um=YY)
print("written:", (OUT_DIR / f"{H5_FILE.stem}_maps.npz").name,
      "and", (OUT_DIR / f"{H5_FILE.stem}_elemental_maps.png").name)

## Next steps

2. **Peak shape.** The residual is dominated by low-energy tailing on the
   strongest lines. A Hypermet tail/shelf term addresses this; a free Lorentzian
   gamma does not, it just broadens everything.
3. **Pile-up.** At a dead-time fraction of 0.6-0.8 sum peaks are expected. The
   2x(Ir La) sum peak is visible near channel 1830, outside this ROI.
4. **The emission side**, which is now the blocking item. Section 6 shows the
   excitation side alone moves the L2/L1 ratios by 20-50 % in the wrong
   direction, so detector efficiency and self-absorption have to be modelled
   together with Phi(E), not after it. Then secondary fluorescence, before any
   group area is read as a concentration.
5. **Energy calibration** as a fitted object with reported residuals, replacing
   the fixed gain/offset used here.